# Customer Segmentation using K-Means & Hierarchical Clustering

**Problem Statement:** Implement K-Means clustering / Hierarchical clustering on the dataset (`sales_data_sample.csv` / Mall Customers dataset) and determine the optimal number of clusters using the **Elbow Method**.

**Dataset Features:**
1. `CustomerID`
2. `Gender` (Customer Gender)
3. `Age` (Customer Age)
4. `Annual Income (k$)` (Annual Income in Thousand Dollars)
5. `Spending Score (1-100)` (Based on customer behavior and spending nature)

### Step 1: Import Required Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

### Step 2: Load the Dataset
*Note: Automatically checks locally for `sales_data_sample.csv` or `Mall_Customers.csv` first (note that the 5 features listed in the problem statement belong to the classic Mall Customers dataset, often renamed `sales_data_sample.csv` in lab manuals). Also handles the classic `SALES` dataset if your lab uses that file, and includes online + offline fallbacks.*

In [2]:
local_files = ['sales_data_sample.csv', 'Mall_Customers.csv', 'mall_customers.csv', 'customers.csv']
df = None

# 1. Check local directory first
for f in local_files:
    if os.path.exists(f):
        try:
            df = pd.read_csv(f)
        except UnicodeDecodeError:
            df = pd.read_csv(f, encoding='latin1')
        print(f"Loaded local dataset: {f}")
        break

# 2. If not local, load the exact 5-feature Mall Customers dataset from online mirror
if df is None:
    urls = [
        "https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv",
        "https://raw.githubusercontent.com/kennedykwangari/Mall-Customer-Segmentation-Data/master/Mall_Customers.csv"
    ]
    for url in urls:
        try:
            df = pd.read_csv(url)
            print("Loaded dataset directly from online mirror!")
            break
        except Exception:
            continue

# 3. Offline fallback if lab PC has neither CSV nor internet access
if df is None:
    print("Offline mode detected & CSV not found locally. Generating realistic synthetic dataset (200 rows)...")
    centers = [(25, 20), (25, 80), (55, 50), (85, 20), (85, 80)]
    inc, spend = [], []
    for cx, cy in centers:
        inc.extend(np.clip(np.random.normal(cx, 8, 40).astype(int), 15, 137))
        spend.extend(np.clip(np.random.normal(cy, 9, 40).astype(int), 1, 99))
    df = pd.DataFrame({
        'CustomerID': np.arange(1, 201),
        'Gender': np.random.choice(['Male', 'Female'], size=200),
        'Age': np.random.randint(18, 70, size=200),
        'Annual Income (k$)': inc,
        'Spending Score (1-100)': spend
    })

print("Dataset Shape:", df.shape)
df.head()

Loaded dataset directly from online mirror!
Dataset Shape: (200, 5)


,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


### Step 3: Exploratory Data Analysis & Feature Selection
- Automatically selects `Annual Income (k$)` and `Spending Score (1-100)` (or numeric sales columns if the classic Kaggle `sales_data_sample.csv` is loaded locally).

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nStatistical Summary:\n", df.describe())

# Automatically detect columns (supports both the 5-feature Customer dataset and classic sales_data_sample.csv)
if 'Annual Income (k$)' in df.columns and 'Spending Score (1-100)' in df.columns:
    feature_cols = ['Annual Income (k$)', 'Spending Score (1-100)']
elif 'SALES' in df.columns and 'PRICEEACH' in df.columns:
    feature_cols = ['QUANTITYORDERED', 'SALES']
else:
    # Fallback to last two numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = numeric_cols[-2:]

print("\nSelected Features for Clustering:", feature_cols)
X = df[feature_cols].values

# Standardize features for distance-based clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Step 4: Determine Optimal Number of Clusters using the Elbow Method (WCSS)

In [ ]:
wcss = []  # Within-Cluster Sum of Squares
k_range = range(1, 11)

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans_temp.fit(X_scaled)
    wcss.append(kmeans_temp.inertia_)

# Plot the Elbow Graph
plt.figure(figsize=(8, 4))
plt.plot(k_range, wcss, marker='o', linestyle='--', color='navy')
plt.title('The Elbow Method for Optimal Number of Clusters (K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.show()

# From the elbow curve, optimal K = 5 for customer income vs spending segmentation
optimal_k = 5
print(f"Optimal number of clusters determined from the Elbow curve: K = {optimal_k}")

### Step 5: Implement K-Means Clustering & Visualize Clusters

In [ ]:
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X_scaled)
df['KMeans_Cluster'] = y_kmeans

sil_kmeans = silhouette_score(X_scaled, y_kmeans)
print(f"K-Means Silhouette Score (K={optimal_k}): {sil_kmeans:.4f}")

# Inverse transform centroids back to original scale for plotting
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

plt.figure(figsize=(9, 6))
colors = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4']
for i in range(optimal_k):
    plt.scatter(X[y_kmeans == i, 0], X[y_kmeans == i, 1], s=60, c=colors[i], label=f'Cluster {i+1}')

plt.scatter(centroids[:, 0], centroids[:, 1], s=250, c='gold', edgecolors='black', marker='*', label='Centroids')
plt.title(f'Customer Segments using K-Means Clustering (K = {optimal_k})')
plt.xlabel(feature_cols[0])
plt.ylabel(feature_cols[1])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Step 6: Implement Hierarchical (Agglomerative) Clustering & Dendrogram

In [ ]:
# 1. Plot Dendrogram using Ward linkage
plt.figure(figsize=(10, 5))
dendrogram = sch.dendrogram(sch.linkage(X_scaled, method='ward'))
plt.title('Dendrogram (Hierarchical Clustering - Ward Linkage)')
plt.xlabel('Customers')
plt.ylabel('Euclidean Distances')
plt.axhline(y=10, color='r', linestyle='--', label='Cut Threshold (5 Clusters)')
plt.legend()
plt.show()

# 2. Fit Agglomerative Hierarchical Clustering
hc = AgglomerativeClustering(n_clusters=optimal_k, metric='euclidean', linkage='ward')
y_hc = hc.fit_predict(X_scaled)
df['Hierarchical_Cluster'] = y_hc

sil_hc = silhouette_score(X_scaled, y_hc)
print(f"Hierarchical Clustering Silhouette Score (K={optimal_k}): {sil_hc:.4f}")

# 3. Visualize Hierarchical Clusters
plt.figure(figsize=(9, 6))
for i in range(optimal_k):
    plt.scatter(X[y_hc == i, 0], X[y_hc == i, 1], s=60, c=colors[i], label=f'Cluster {i+1}')

plt.title(f'Customer Segments using Hierarchical Clustering (K = {optimal_k})')
plt.xlabel(feature_cols[0])
plt.ylabel(feature_cols[1])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Step 7: Cluster Summary Table (Viva / Analysis)

In [ ]:
print("Cluster-wise Mean Values (K-Means):")
df.groupby('KMeans_Cluster')[feature_cols].agg(['count', 'mean']).round(2)